# 11 — Shot Noise & Your First Circuit

Turn the one-qubit toolkit of `01/01`–`01/10` into something a quantum computer can *run*: write a circuit, let a simulated device execute it shot by shot, read the counts, and check that everything you derived — the Born shadow, the native gates, the error bar — shows up in the data. Then hear the first honest complaint of a real machine.

**Prerequisites:** `01/08_hadamard_and_measurement_bases` ($H$, rotate-then-measure), `01/09_phase_and_rotation_gates` (rotations, state preparation, native gates), `01/10_expectation_values` (the shot estimator and its error bar), `01/04_measurement_and_born_rule`.

**What you'll be able to do**
- Write, draw and run a one-qubit **circuit** in Qiskit — qubit register, gates in time order, measurement into a classical bit — and read its **counts** as a sample of the Born shadow.
- See what the device *actually* executes: the compiler rewrites $H$ into the native gates $R_z$ and $\sqrt X$ of `01/09`, and you will check that rewrite by hand.
- Run the three-axis recipe of `01/10` as three circuits, each estimate with its error bar.
- Diagnose, on a simulator that carries a real device's noise, an error that **no number of shots** removes, trace it to its source, and say exactly what kind of uncertainty a state vector cannot hold — the honest reason the next notebook introduces the density matrix.

You have the pieces: a qubit is two amplitudes (`01/01`), gates are unitaries (`01/06`), $H$ builds $|+\rangle$ (`01/08`), and `01/10` told you exactly how noisy $N$ shots of a measurement are. Nothing in this notebook is new *physics* — it is your theory meeting a machine, and the pleasure of seeing it hold.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import Operator, Statevector
from qiskit_aer import AerSimulator

from qot_course import viz
from qot_course.colors import COLORS
from qot_course.quantum.states import KET_0, KET_PLUS, qubit_state, born_probabilities, bloch_vector
from qot_course.quantum.gates import PAULI_X, PAULI_Y, PAULI_Z, HADAMARD, S_DAG, expectation

rng = np.random.default_rng(0)   # seeds for the simulator runs and the exercises
viz.use_course_style()

## 1. Your first circuit: write it, predict it, run it

A **quantum circuit** is the program a quantum computer runs. It has three parts, read left to right in time:

1. a register of qubits, each starting in $|0\rangle$;
2. a sequence of **gates** — the unitaries of `01/06`–`01/09` — applied in order;
3. **measurements** that write each qubit's outcome, a $0$ or a $1$, into a classical bit.

Running the circuit once gives one bit string. Running it $N$ times — $N$ **shots** — gives **counts**: how many times each string came out. That is all a device ever returns.

Every gate you met has a one-line spelling: the Paulis of `01/07` are `qc.x(0)`, `qc.y(0)`, `qc.z(0)`, the Hadamard of `01/08` is `qc.h(0)`, $S^\dagger$ is `qc.sdg(0)`, the dials of `01/09` are `qc.rx/ry/rz(theta, 0)`. And a circuit *is* the product of its gate matrices, rightmost applied first — `01/06` exercise 3 promised you would meet that here.

The smallest interesting program: one qubit, one gate, one measurement. Apply $H$ to $|0\rangle$ and read the qubit. You can predict the result completely before running anything. By hand, $H|0\rangle = \tfrac{1}{\sqrt2}(|0\rangle + |1\rangle) = |+\rangle$ (`01/08`: the first column of $H$), and the Born rule (`01/04`) gives $P(0) = P(1) = \tfrac12$. From `01/10`, the estimate $\widehat{\langle Z\rangle} = (n_0 - n_1)/N$ of $N$ shots will sit within about $\sigma_N = \sqrt{(1 - 0^2)/N} = 1/\sqrt N$ of the exact $\langle Z\rangle = 0$ — the worst case, because this is a fair coin. Let's build it.

In [ ]:
qc = QuantumCircuit(1, 1)     # one qubit (starts in |0>), one classical bit to hold the outcome
qc.h(0)                        # H on qubit 0:  |0> -> |+>
qc.measure(0, 0)               # read qubit 0 in the Z basis, store the result in classical bit 0
qc.draw("mpl")

**Read the figure.** The wire `q` is the qubit, running left to right in time; `c` is the classical bit. The box `H` is the gate, and the meter is the measurement, with the arrow down to the classical wire showing where the outcome lands. (Qiskit numbers qubits and bits from $0$; with several qubits it prints bit strings right-to-left, bit $0$ last — the two-qubit notebooks from `01/17` onward rely on that convention; today one bit is enough.)

Before the measurement the circuit is a unitary acting on $|0\rangle$, so it *has* a state vector — the same object you worked with all module long. Let's ask Qiskit for it and compare with `KET_PLUS`; then run the full circuit with measurement on a simulator and compare the counts with the Born prediction.

In [ ]:
# the state right before the meter: the circuit without its measurement, applied to |0>
state_before_measure = Statevector.from_instruction(qc.remove_final_measurements(inplace=False)).data
print("circuit state before measurement :", np.round(state_before_measure, 4))
print("KET_PLUS from 01/01               :", np.round(KET_PLUS, 4))
print("Born prediction (01/04)           :", born_probabilities(state_before_measure))
print("matrix of the circuit == HADAMARD :", np.allclose(Operator(qc.remove_final_measurements(inplace=False)).data, HADAMARD), "  (a circuit is the product of its gates, 01/06)")

# now run it shot by shot on a simulated device
sim = AerSimulator()
N = 2048
counts = sim.run(transpile(qc, sim), shots=N, seed_simulator=int(rng.integers(2**31))).result().get_counts()
print(f"\ncounts from {N} shots            :", counts)

n0, n1 = counts.get("0", 0), counts.get("1", 0)
z_hat, sigma = (n0 - n1) / N, 1 / np.sqrt(N)
print(f"estimated <Z> = {z_hat:+.4f} +- {sigma:.4f}   (exact 0: off by {abs(z_hat) / sigma:.1f} sigma)")

viz.plot_counts(counts)
plt.title(f"|+> measured {N} times on a simulator", pad=12)
plt.show()

**Read the output and the figure.** The circuit's state is `KET_PLUS` to the digit — the program and the vector are the same object, one written as gates, one as amplitudes. The counts are *not* $1024/1024$: they are a sample of the Born shadow, and the estimated $\langle Z\rangle$ misses $0$ by a fraction of the $1/\sqrt N$ error bar `01/10` predicted for a fair coin. Keys of the `counts` dictionary are bit strings, values are integers — data, not probabilities; dividing by $N$ gives the *estimated* probabilities, and the error bar says how much to trust them. This is the error bar of `01/10` holding on a simulated device; `01/10` tested it against repeated trials, and exercise 1 asks you to repeat that test here.

Now let's watch the shadow sharpen. The same circuit at $20$, $200$ and $2000$ shots:

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4.2))
for ax, shots in zip(axes, [20, 200, 2000]):
    c = sim.run(transpile(qc, sim), shots=shots, seed_simulator=int(rng.integers(2**31))).result().get_counts()
    viz.plot_counts(c, ax=ax)
    p0_hat = c.get("0", 0) / shots
    ax.set_title(f"{shots} shots: P(0) = {p0_hat:.3f} +- {1 / (2 * np.sqrt(shots)):.3f}", pad=10)
fig.suptitle("One circuit, three budgets: the counts converge toward the 50/50 Born shadow", fontsize=15, fontweight="bold")
fig.tight_layout()
plt.show()

**Read the figure.** At $20$ shots the split can be visibly lopsided — pure chance, nothing is wrong; by $2000$ shots it hugs $50/50$. The error bar on $\widehat{P}(0)$ shown in each title is $\sigma_N/2$ (since $P(0) = (1 + \langle Z\rangle)/2$, `01/04`), and it shrinks by $\sqrt{10}$ from panel to panel. This is **shot noise** as you derived it in `01/10` — here it arrives through a circuit instead of a NumPy sampler. Precision costs repetitions, at a square-root rate, and you can price it in advance.

## 2. What the machine really executes: native gates

You wrote `qc.h(0)`, but a device does not own an $H$ button. `01/09` mentioned the gates IBM machines execute natively on one qubit: $R_z(\theta)$ for any angle and a fixed $\sqrt X = R_x(\pi/2)$ up to phase. The full native set adds $X$ and one fixed two-qubit entangler: CNOT on the Manila-era chips simulated here, ECR on IBM's Eagle chips and CZ on its Heron chips. New today: the $R_z$ is not even a pulse. It is applied in software, by shifting the phase reference of every later pulse, so it is essentially free and exact, and the $\sqrt X$ is the one physical single-qubit pulse. The **transpiler** rewrites your circuit into that native set. Let's look at what it produces for $H$, and then check the rewrite the way `01/09` taught you: multiply the matrices and compare up to a global phase.

In [ ]:
NATIVE = ["rz", "sx", "x", "cx"]                      # IBM native set of the Manila-era chips (Eagle chips: ecr instead of cx; Heron chips: cz)
qc_native = transpile(qc, basis_gates=NATIVE, optimization_level=1)
print(qc_native.draw("text"))
print("native gate counts:", dict(qc_native.count_ops()))

# check the rewrite by hand: the product of the native matrices must be H up to a global phase
SX = np.array([[1 + 1j, 1 - 1j], [1 - 1j, 1 + 1j]]) / 2            # sqrt(X): SX @ SX = X
R_Z = lambda t: np.array([[np.exp(-1j * t / 2), 0], [0, np.exp(1j * t / 2)]])
product = R_Z(np.pi / 2) @ SX @ R_Z(np.pi / 2)                      # gates act right-to-left: first Rz, then SX, then Rz
print("\nSX @ SX == X :", np.allclose(SX @ SX, PAULI_X))
ratio = product / HADAMARD
print("product / H  =", np.round(ratio, 4).tolist(), "  -> one common factor = a global phase of", np.round(np.angle(ratio[0, 0]) / np.pi, 3), "pi")
print("equal up to global phase:", np.allclose(product, ratio[0, 0] * HADAMARD))

# the same check through Qiskit's own matrix of the compiled circuit (measurement removed)
U_native = Operator(qc_native.remove_final_measurements(inplace=False)).data
print("Operator(compiled circuit) equals H up to phase:", np.allclose(U_native / (U_native[0, 0] / HADAMARD[0, 0]), HADAMARD))

**Read the output.** One $H$ became three native operations, $R_z(\pi/2)\,\sqrt X\,R_z(\pi/2)$, and the hand product of those three matrices is $H$ times a single common factor $e^{-i\pi/4}$ — a global phase (Qiskit records the compensating $\pi/4$ as the circuit's "global phase" in the drawing), which `01/03` and `01/06` told you no measurement can see. This is the three-rotation theorem of `01/09` in native clothing — with the middle rotation about $x$ instead of $y$ ($\sqrt X \propto R_x(\pi/2)$), a $Z$–$X$–$Z$ form equivalent to the $Z$–$Y$–$Z$ one. The compiler's job is exactly the one you could do by hand: find native gates whose product equals your gate *up to a phase*. It matters because on real hardware every native gate costs a little error (`05/03`), so the circuit that runs — not the one you wrote — sets the error budget.

## 3. Three circuits, one state: the tomography recipe as programs

`01/10` §3 estimated $\langle X\rangle, \langle Y\rangle, \langle Z\rangle$ of the tilted state by rotating each basis onto $Z$ and averaging. Here the state is prepared by gates, the way `01/09` taught: $R_z(\phi)R_y(\theta)|0\rangle$. As circuits, that recipe is three programs that differ only in their last gates before the meter: nothing for $Z$, `h` for $X$, `sdg` then `h` for $Y$ (`01/08`). This is *exactly* what `01/15` will run on a noisy backend to reconstruct a density matrix — here, on the ideal simulator, the three estimates should land on the Bloch vector of `01/03` within their error bars.

In [ ]:
theta_t, phi_t = np.pi / 3, np.pi / 5                  # the tilted state of 01/10, now prepared by gates
psi = qubit_state(theta=theta_t, phi=phi_t)

def run_counts(circuit, shots, backend=sim):
    """Compile for the backend, run `shots` times, return the counts dict."""
    return backend.run(transpile(circuit, backend), shots=shots, seed_simulator=int(rng.integers(2**31))).result().get_counts()

def estimate_Z_from_counts(counts):
    """(n0 - n1)/N and its error bar sqrt((1 - est^2)/N), computed from the data as a lab would."""
    n0, n1 = counts.get("0", 0), counts.get("1", 0)
    est = (n0 - n1) / (n0 + n1)
    return est, np.sqrt((1 - est**2) / (n0 + n1))

N = 4000
def tilted_circuit_in_basis(axis):
    """Prepare the tilted state, rotate the chosen Pauli basis onto Z (01/08), then measure."""
    c = QuantumCircuit(1, 1)
    c.ry(theta_t, 0); c.rz(phi_t, 0)          # state preparation
    if axis == "X":
        c.h(0)                                # X basis -> Z
    elif axis == "Y":
        c.sdg(0); c.h(0)                      # Y basis -> Z
    c.measure(0, 0)
    return c

estimates, errors = {}, {}
for axis, P in zip("XYZ", (PAULI_X, PAULI_Y, PAULI_Z)):
    estimates[axis], errors[axis] = estimate_Z_from_counts(run_counts(tilted_circuit_in_basis(axis), N))
    exact = expectation(psi, P)
    print(f"<{axis}>: circuit estimate {estimates[axis]:+.3f} +- {errors[axis]:.3f}   exact {exact:+.3f}   off by {abs(estimates[axis] - exact) / errors[axis]:.1f} sigma")

# sanity: the preparation circuit gives the state we think it does
prep_state = Statevector.from_instruction(tilted_circuit_in_basis("Z").remove_final_measurements(inplace=False)).data
print("\n|<circuit state | qubit_state>| =", round(abs(np.vdot(prep_state, psi)), 6), "  -> the same state up to a global phase (01/03)")

r_hat = np.array([estimates[a] for a in "XYZ"])
print("\nestimated Bloch vector :", np.round(r_hat, 3), "  length", round(float(np.linalg.norm(r_hat)), 3))
print("exact Bloch vector     :", np.round(bloch_vector(psi), 3), "  length 1.000 (pure state)")
# a circuit is a product of matrices, rightmost first (01/06 ex. 3): the Y circuit is H S^dag R_z(phi) R_y(theta)
R_Y = lambda t: np.array([[np.cos(t / 2), -np.sin(t / 2)], [np.sin(t / 2), np.cos(t / 2)]], dtype=complex)
U_Y = Operator(tilted_circuit_in_basis("Y").remove_final_measurements(inplace=False)).data
print("\nOperator(Y circuit) == H @ S_DAG @ R_z(phi) @ R_y(theta):", np.allclose(U_Y, HADAMARD @ S_DAG @ R_Z(phi_t) @ R_Y(theta_t)))
print("the Y circuit, compiled:")
print(transpile(tilted_circuit_in_basis("Y"), basis_gates=NATIVE, optimization_level=1).draw("text"))

**Read the output.** The circuit's state differs from the `qubit_state` of `01/10` by a global phase only (the overlap has modulus $1$), so it is the same point on the Bloch sphere (`01/03`). Each Pauli average is recovered by its own circuit to within its error bar, and the three together reconstruct a Bloch vector of length close to $1$ — the state is pure, and the ideal simulator knows it. The matrix of the $Y$ circuit is the product $H\,S^\dagger\,R_z(\phi)\,R_y(\theta)$, rightmost gate first — a circuit is a product of unitaries, as `01/06` said. Notice the compiled $Y$ circuit: state preparation, $S^\dagger$ and $H$ have all melted into a short string of $R_z$ and $\sqrt X$ — the compiler merges neighbouring rotations (`01/09`'s $R_z R_y R_z$ says any one-qubit circuit needs at most two $\sqrt X$). Keep this picture: three circuits, three averages, one state. `01/15` reruns it with noise and finds a length *below* $1$ — and the rest of this notebook explains why that is not a fluke of shot noise.

## 4. The first honest complaint of a real machine

Everything above had one message: with enough shots, the data agrees with the state vector. Here is the complaint. `01/10` said an eigenstate of $Z$ is the cheapest state there is: $\langle Z\rangle = \pm1$, $\sigma_N = 0$, a single shot settles it. So let's run the *emptiest* circuit — prepare $|0\rangle$, do nothing, measure — on a simulator that carries the **noise model of a real IBM device**, built from a calibration snapshot of that device: readout errors, gate errors and decay times measured once by IBM and frozen into the software (`05/01` introduces such backends). It is a record of past measurements, not a live device. Shot noise predicts the count of $1$s should be exactly zero. Watch what happens, and watch what happens as $N$ grows.

In [ ]:
from qot_course.hardware.runtime import get_noisy_backend

noisy = get_noisy_backend("FakeManilaV2")  # AerSimulator carrying the FakeManila calibration snapshot
print("backend:", noisy.name)
# get_noisy_backend falls back to an ideal simulator if the snapshot cannot load: stop here rather than read noiseless counts
assert noisy.options.noise_model is not None, "no noise model loaded: the cells below would show no floor"

qc_zero = QuantumCircuit(1, 1); qc_zero.measure(0, 0)     # prepare |0> (do nothing), read it
budgets = np.array([256, 1024, 4096, 16384, 65536])
miss, bar = [], []
for N in budgets:
    est, err = estimate_Z_from_counts(run_counts(qc_zero, int(N), backend=noisy))
    miss.append(1 - est); bar.append(1 / np.sqrt(N))
    print(f"N = {N:6d}: estimated <Z> = {est:.4f}  (exact for |0>: 1.0000;  shot-noise worst case 1/sqrt(N) = {1/np.sqrt(N):.4f};  miss = {1 - est:.4f})")

# the three-axis recipe of section 3 on the noisy |0>: is there ANY pure state behind these counts?
def zero_circuit_in_basis(axis):
    c = QuantumCircuit(1, 1)
    if axis == "X":
        c.h(0)
    elif axis == "Y":
        c.sdg(0); c.h(0)
    c.measure(0, 0)
    return c

r_noisy = np.array([estimate_Z_from_counts(run_counts(zero_circuit_in_basis(a), 65536, backend=noisy))[0] for a in "XYZ"])
print(f"\nnoisy |0>, three axes at 65536 shots: (<X>,<Y>,<Z>) = {np.round(r_noisy, 4)},  length = {np.linalg.norm(r_noisy):.4f}  (every pure state has length 1.0000 +- {1/np.sqrt(65536):.4f})")

# which physical qubit did the compiler pick, and what does the device's calibration say about its meter?
from qiskit_ibm_runtime.fake_provider import FakeManilaV2
phys = transpile(qc_zero, noisy).layout.initial_index_layout()[0]
readout_err = FakeManilaV2().target["measure"][(phys,)].error
t1_us = FakeManilaV2().qubit_properties(phys).t1 * 1e6
print(f"compiler placed the circuit on physical qubit {phys}; its calibrated readout error is {readout_err:.4f}, its T1 {t1_us:.0f} us (snapshot values)")
p1_obs = miss[-1] / 2
print(f"observed P(1 | prepared 0) at 65536 shots = {p1_obs:.4f}   (= miss / 2, since P(1) = (1 - <Z>)/2)")
print(f"binomial standard error of that P(1): {np.sqrt(p1_obs * (1 - p1_obs) / 65536):.4f};  |observed - calibrated| = {abs(p1_obs - readout_err):.4f}")

# the discriminating run: same compiled circuit, same seed, but a noise model that keeps ONLY the readout confusion
from qiskit_aer.noise import NoiseModel
readout_only = AerSimulator(noise_model=NoiseModel.from_backend(FakeManilaV2(), gate_error=False, thermal_relaxation=False))
compiled = transpile(qc_zero, noisy)
seed = int(rng.integers(2**31))
c_full = noisy.run(compiled, shots=65536, seed_simulator=seed).result().get_counts()
c_readout = readout_only.run(compiled, shots=65536, seed_simulator=seed).result().get_counts()
print("full device noise model :", c_full)
print("readout confusion only  :", c_readout, "  identical:", c_full == c_readout)

# the same discriminating run on a circuit WITH a gate: x, then measure. Now gate error and decay have something to act on
qc_one = QuantumCircuit(1, 1); qc_one.x(0); qc_one.measure(0, 0)
compiled_one = transpile(qc_one, noisy)
c_full_x = noisy.run(compiled_one, shots=65536, seed_simulator=seed).result().get_counts()
c_readout_x = readout_only.run(compiled_one, shots=65536, seed_simulator=seed).result().get_counts()
p0_full, p0_ro = c_full_x.get("0", 0) / 65536, c_readout_x.get("0", 0) / 65536
se_gap = np.sqrt(p0_full * (1 - p0_full) / 65536 + p0_ro * (1 - p0_ro) / 65536)
print(f"\nX|0>, full device noise model : P(0) = {p0_full:.4f}")
print(f"X|0>, readout confusion only  : P(0) = {p0_ro:.4f}   gap = {p0_full - p0_ro:+.4f}  (standard error of the gap {se_gap:.4f})   identical: {c_full_x == c_readout_x}")
print(f"calibrated error of one x gate on qubit {phys}: {FakeManilaV2().target['x'][(phys,)].error:.5f}")

fig, ax = plt.subplots(figsize=(7.0, 4.2))
ax.loglog(budgets, bar, "--", color=COLORS["muted"], lw=1.5, label=r"shot-noise worst case $1/\sqrt{N}$ (ideal device: miss = 0)")
ax.loglog(budgets, miss, "o-", color=COLORS["highlight"], lw=2, ms=8, label=r"miss $1 - \widehat{\langle Z\rangle}$ of $|0\rangle$ on the noisy device")
ax.set_xlabel("number of shots N"); ax.set_ylabel("distance from the exact <Z> = 1")
ax.set_title("A floor that more shots cannot buy down", pad=12)
ax.legend(fontsize=8)
plt.show()

**Read the output and the figure.** Prepared in $|0\rangle$ and read immediately, the noisy device still reports a $1$ about $1$–$2\%$ of the time, and the miss $1 - \widehat{\langle Z\rangle}$ does **not** shrink with $N$: it sits flat while the shot-noise line falls away beneath it (the first point, at $N = 256$, wobbles by about one of its own error bars — there the floor and the noise are still the same size). By $N = 65536$ the miss is almost eight times even the worst-case bar $1/\sqrt N$ — and `01/10` predicted $\sigma_N = 0$ for an eigenstate, so *any* miss at all is already outside the theory of shot noise. No error bar of `01/10`'s kind explains it, because that error bar describes the *scatter* of an estimate around the truth, not a *systematic* offset.

Be careful about what this does and does not show — it is worth being exact. A lopsided $Z$ histogram alone proves nothing about purity: a perfectly pure tilted state $R_y(\theta)|0\rangle$ gives any split you like (exercise 2), and `01/04` warned that one shadow cannot tell states apart. That is why the code measures all three axes: every pure state has a Bloch vector of length exactly $1$ (`01/03`, `01/10` §3), and the three-axis recipe on the noisy $|0\rangle$ returns a length of about $0.97$, almost eight error bars short. **No pure state read by a perfect meter produces these counts.** Then the last lines name the culprit *for this circuit*: the compiler placed it on the physical qubit with the best meter, that meter's calibrated misassignment probability — the chance of reading $1$ when the qubit is in $|0\rangle$ — matches the observed $P(1)$ within its sampling error (they differ by $0.0006$, about one binomial standard error of $0.0005$), and the discriminating run settles it: the same compiled circuit, the same seed, with every gate error and every decay switched *off* and only the readout confusion kept, returns **identical counts**. On the emptiest circuit, in this noise model, the floor is entirely the **meter**: a classical confusion on the record, which leaves the qubit's state untouched and which module 05 calibrates and corrects (`05/02`). So this floor, by itself, does not yet force a new description of the *state*.

The emptiest circuit is the one where only the meter can act, so the last lines repeat the discriminating run on a circuit with one gate, `x` then measure. Now the gate error and the decay have something to act on, and the counts of the two models are no longer identical. Yet the gap in $P(0)$ is $+0.0001$, far below its standard error of $0.0007$: one gate whose calibrated error is $0.00021$ adds too little to see beside the meter's confusion. Gate error and decay need longer circuits, or waiting time, to show; `01/15` adds idle time before the meter and watches them take over.

Here is what does. A real qubit does not only get misread — between preparation and readout it *changes*: it decays toward $|0\rangle$ (with a characteristic time $T_1$, $179\,\mu$s for this qubit in the calibration snapshot), it loses its phase, its gates land slightly off. After such an event the honest description of the qubit is *"probably still $|1\rangle$, with some probability already $|0\rangle$"* — and $a_0|0\rangle + a_1|1\rangle$ has no slot for "with some probability". A vector can be tilted; it cannot be *uncertain*. That slot is the density matrix $\rho$ of `01/12`; the length-below-one of a Bloch vector becomes the **purity** of `01/13`; `01/15` reruns today's three circuits on this same noisy backend, with gates that can go wrong, and reconstructs $\rho$; `01/19` gives the decay itself a name (a *channel*) and `01/20` puts it on a clock; and `05/02` meets the same floor on a real machine. What you take from today is the diagnostic: a miss that does not shrink with $N$ is systematic, and a Bloch vector shorter than $1$ says the pure-state-plus-perfect-meter story has ended.

For the curious, here is the short idiom that runs this circuit on a real IBM QPU; Module 05 runs it, starting in `05/01`.

```python
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2
service = QiskitRuntimeService()                          # uses your saved account (no token in code)
backend = service.least_busy(operational=True, simulator=False)
sampler = SamplerV2(mode=backend)
job = sampler.run([transpile(qc, backend)])
counts = job.result()[0].data.c.get_counts()
```

## Your turn

1. **Watch the noise shrink.** Re-run the three-panel counts figure of §1 with `shots = [5, 50, 500, 5000]`, and for each panel print $\widehat{P}(0) \pm \sigma_N/2$. Over how many panels does the estimate stay within one error bar of $\tfrac12$? Is that about what a $\pm1\sigma$ bar promises (`01/10`: roughly two times in three)? Then repeat the $500$-shot run $100$ times with fresh seeds from `rng` and compare the spread of the $100$ estimates of $P(0)$ with $\sigma_N/2$: the repeated-trial test of `01/10` §2, now on a simulated device.
2. **A biased circuit.** Replace `qc.h(0)` by `qc.ry(theta, 0)` for a $\theta$ of your choosing, predict $P(0) = \cos^2(\theta/2)$ from `01/04`/`01/09`, run $1000$ shots and quote $\widehat{P}(0)$ with its error bar. Which $\theta$ makes the error bar largest, and which makes it vanish? Check your answer against $\sigma_N = \sqrt{(1 - \langle Z\rangle^2)/N}$.
3. **Build $|-\rangle$ and tell it apart.** Apply `h` then `z` to $|0\rangle$ and measure: which counts do you get, and why can a $Z$ measurement not tell $|-\rangle$ from $|+\rangle$ (`01/04`'s phase-blindness)? Now insert `h` before the meter (`01/08`, rotate-then-measure): which counts distinguish the two states, and what does each histogram look like?
4. **Harder: how big is a native circuit?** Transpile `ry(theta)` alone to the native set for five random $\theta$ drawn with `rng`: what sequence comes out, and does `Operator(...)` equal the $R_y(\theta)$ matrix of `01/09` up to a global phase? Then build $H \cdot S \cdot H \cdot T \cdot H$ (five gates) and transpile it with `optimization_level=3`: how many native gates remain? Explain the count from the three-rotation theorem of `01/09` (Nielsen & Chuang Thm 4.1): any one-qubit unitary needs at most two $\sqrt X$ and three $R_z$ (hint: check that $\sqrt X\,R_z(\gamma)\,\sqrt X$ is, up to phase and outer $R_z$'s, a rotation by an angle set by $\gamma$ — so two *fixed* $\sqrt X$ plus three *variable* $R_z$ reach any $U$).

## What you built

- You wrote, drew and ran a one-qubit circuit, read its state before the meter, and read its counts as a sample of the Born shadow — with the error bar `01/10` predicted.
- You saw the compiler rewrite $H$ into $R_z(\pi/2)\,\sqrt X\,R_z(\pi/2)$ and checked the rewrite by hand, up to a global phase — the native gates of `01/09`, the circuit that really runs.
- You ran the three-axis tomography recipe of `01/10` as three circuits, each estimate with its error bar.
- You met a systematic floor on a noisy device that no number of shots removes, measured all three axes to show that no pure state read by a perfect meter produces those counts, traced this particular floor to the meter's calibrated misassignment, saw that one gate adds too little to measure beside it, and can now say which kind of uncertainty a state vector cannot describe.
- Your circuit has a classical cousin: a randomized program on bits, which initializes its bits, applies logic gates in order and reads the output.
- You can predict what repeating that classical program gives: run it many times and the output frequencies you estimate carry sampling error, exactly as your shot counts do.

The theory of `01/01`–`01/10` runs as a program, and the data obeys it — until a real device asks for a richer description. That is not a crack in what you learned; it is the door to the next ten notebooks, and you opened it with your own counts.

**New in the dictionary.** The complete table lives in [the course dictionary](../dictionary.ipynb) (`notebooks/dictionary.ipynb`).

| Classical | Quantum |
| --- | --- |
| a randomized program on bits, run many times to estimate its output frequencies | a circuit run for $N$ shots, whose counts sample the Born shadow |

## What's next

A noisy device holds *"probably $|0\rangle$, possibly something else"*, and no state vector can say that. In `01/12_from_states_to_density_matrix` we build the object that can — the density matrix $\rho = |\psi\rangle\langle\psi|$ for a pure state first, then, in `01/13`, the mixtures that the floor of §4 demands — the central character of the rest of the course.

## References

- M. A. Nielsen & I. L. Chuang, *Quantum Computation and Quantum Information*, §1.3 (quantum circuits), §4.2 (single-qubit operations, Thm 4.1), §4.4 (measurement), Cambridge University Press (2010). doi:10.1017/CBO9780511976667
- A. Javadi-Abhari et al., *Quantum computing with Qiskit* (2024). arXiv:2405.08810. doi:10.48550/arXiv.2405.08810
- D. C. McKay, C. J. Wood, S. Sheldon, J. M. Chow & J. M. Gambetta, *Efficient Z gates for quantum computing*, Phys. Rev. A **96**, 022330 (2017). doi:10.1103/PhysRevA.96.022330 — why $R_z$ is "free" (virtual) on IBM hardware and $\sqrt X$ is the one physical single-qubit pulse.

**Previous:** `notebooks/01_QuantumFoundations/10_expectation_values.ipynb` · **Next:** `notebooks/01_QuantumFoundations/12_from_states_to_density_matrix.ipynb`